In [ ]:
import os # necesario para leer rutas y variables de entorno
import time # necesario para el sleep para que se note el typing y esperar el remove.webhook de telegram
import csv # necesario para guardar el historial de mensajes
import json # necesario para request del dolar
import requests # necesario para conexion con API de spoonacular y dolar
import telebot # funciones del bot de telegram
from dotenv import load_dotenv # leer variables de entorno del .env
from datetime import datetime # necesario para timestamp del historial

load_dotenv(override=False)
TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
SPOONACULAR_API_KEY = os.getenv("apiKey")

if not TOKEN:
    raise RuntimeError(
        "Falta TELEGRAM_BOT_TOKEN. Creá el archivo .env junto al notebook."
    )
if not SPOONACULAR_API_KEY:
    raise RuntimeError(
        "Falta api key de spoonaculara"
    )

bot = telebot.TeleBot(TOKEN, parse_mode=None)

RUTA_CSV = "historial.csv"

def registrar(message):
    existe = os.path.isfile(RUTA_CSV)

    with open(RUTA_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not existe:
            writer.writerow(["timestamp", "chat_id", "nombre", "mensaje"])

        writer.writerow([
            datetime.now().isoformat(timespec="seconds"),
            message.chat.id,
            message.from_user.first_name or "sin nombre",
            message.text or f"[{message.content_type}]",
        ])

print("Bot creado en Python. Todavía no está escuchando mensajes.")

URL_DOLAR_ACTUAL = "https://api.bluelytics.com.ar/v2/latest"

headers = {
    "User-Agent": "TDI2-BotRecetas/1.0 (trabajo practico; consultas de bajo volumen)"
}


def pedir_json(url, params=None, timeout=20):
    try:
        respuesta = requests.get(url, params=params, headers=headers, timeout=timeout)
        respuesta.raise_for_status()
        return respuesta.json()

    except requests.exceptions.Timeout:
        print("La API tardó demasiado. Probá más tarde.")
    except requests.exceptions.ConnectionError:
        print("No se pudo conectar. Revisá la conexión o el estado de la API.")
    except requests.exceptions.HTTPError as error:
        print(f"La API respondió un error HTTP: {error}")
    except requests.exceptions.JSONDecodeError:
        print("La respuesta no era JSON válido.")

    return None

In [ ]:
# Lo que ve el usuario (izquierda) y lo que entiende Spoonacular (derecha)
INGREDIENTES_SUGERIDOS = {
    "🍗 pollo": "chicken",
    "🍚 arroz": "rice",
    "🍅 tomate": "tomato",
    "🧅 cebolla": "onion",
    "🧄 ajo": "garlic",
    "🥔 papa": "potato",
    "🧀 queso": "cheese",
    "🥚 huevo": "egg",
    "🥛 leche": "milk",
    "🥩 carne": "beef",
}

# Memoria del bot: que ingredientes lleva elegidos cada usuario.
# La clave es el chat_id, el valor es su lista.
ingredientes_por_chat = {}


def teclado_ingredientes(chat_id):
    """Arma los botones, escondiendo los que el usuario ya eligio."""
    teclado = types.InlineKeyboardMarkup(row_width=2)
    elegidos = ingredientes_por_chat.get(chat_id, [])

    botones = []
    for etiqueta, valor in INGREDIENTES_SUGERIDOS.items():
        if valor not in elegidos:
            botones.append(
                types.InlineKeyboardButton(etiqueta, callback_data="ing:" + valor)
            )

    if botones:
        teclado.add(*botones)

    # El boton de buscar aparece recien cuando hay algo elegido
    if elegidos:
        teclado.add(types.InlineKeyboardButton("🔍 Buscar recetas", callback_data="buscar"))

    teclado.add(types.InlineKeyboardButton("❌ Cancelar", callback_data="cancelar"))

    return teclado


def texto_seleccion(chat_id):
    """El mensaje que acompaña a los botones."""
    elegidos = ingredientes_por_chat.get(chat_id, [])

    if not elegidos:
        return "¿Qué tenés en la heladera? Elegí un ingrediente 🥕"

    return (
        "Llevás: <b>" + ", ".join(elegidos) + "</b>\n\n"
        "¿Agregás otro o buscamos ya?"
    )

@bot.message_handler(commands=["start"])
def comando_start(message):
    registrar(message)
    nombre = message.from_user.first_name or "persona"
    texto = (
        f"¡Hola, {nombre}! 👋, soy un bot de ejemplo creado en Python 👾. Me has iniciado 🤖🦾"
    )
    bot.reply_to(message, texto)

@bot.message_handler(commands=["atajos"])
def comando_atajos(message):
    registrar(message)
    teclado = types.InlineKeyboardMarkup(row_width=2)

    btn_start = types.InlineKeyboardButton("/start", callback_data="start")
    btn_dolar = types.InlineKeyboardButton("/dolar", callback_data="dolar")
    btn_receta = types.InlineKeyboardButton("/receta", callback_data="receta")
    btn_historial = types.InlineKeyboardButton("/historial", callback_data="historial")
    
    teclado.add(btn_start, btn_dolar, btn_receta, btn_historial)
    
    texto = "Podes usar estos comandos rápidos:"
    
    bot.reply_to(message, texto, parse_mode="HTML", reply_markup=teclado)

@bot.callback_query_handler(func=lambda call: True)
def manejar_botones(call):
  bot.answer_callback_query(call.id)

  message = call.message
  message.from_user = call.from_user
  chat_id = message.chat.id

  # ---- botones del flujo de ingredientes ----

  if call.data.startswith("ing:"):
    ingrediente = call.data[4:]   # saca el "ing:" del principio
    elegidos = ingredientes_por_chat.setdefault(chat_id, [])

    if ingrediente not in elegidos:
      elegidos.append(ingrediente)

    message.text = "/receta " + ", ".join(elegidos)
    registrar(message)

    # Editamos el mismo mensaje en vez de mandar uno nuevo,
    # asi el chat no se llena de menus repetidos
    bot.edit_message_text(
      texto_seleccion(chat_id),
      chat_id,
      message.message_id,
      parse_mode="HTML",
      reply_markup=teclado_ingredientes(chat_id),
    )
    return

  if call.data == "buscar":
    elegidos = ingredientes_por_chat.pop(chat_id, [])

    if not elegidos:
      bot.send_message(chat_id, "No elegiste ningún ingrediente todavía 🤔")
      return

    message.text = "/receta " + ", ".join(elegidos)
    comando_ingredientes(message)
    return

  if call.data == "cancelar":
    ingredientes_por_chat.pop(chat_id, None)
    bot.send_message(chat_id, "Listo, cancelé la búsqueda ✋")
    return

  # ---- botones de /atajos (lo que ya tenias) ----

  message.text = "/" + call.data

  if call.data == "start":
    comando_start(message)
  elif call.data == "dolar":
    comando_dolar(message)
  elif call.data == "receta":
    comando_ingredientes(message)
  elif call.data == "historial":
    comando_historial(message)

@bot.message_handler(commands=["receta"])
def comando_ingredientes(message):

    registrar(message)

    chat_id = message.chat.id
    partes = message.text.split(maxsplit=1)

# si escribio /receta solo, arranca el menu de botones
    if len(partes) < 2:
        ingredientes_por_chat[chat_id] = []
        bot.send_message(
            chat_id,
            texto_seleccion(chat_id),
            parse_mode="HTML",
            reply_markup=teclado_ingredientes(chat_id),
        )
        return

    ingredientes = partes[1]

    bot.reply_to(
        message,
        "👨‍🍳 Buscando recetas con tus ingredientes..."
    )

    url = "https://api.spoonacular.com/recipes/findByIngredients"

    parametros = {
        "apiKey": SPOONACULAR_API_KEY,
        "ingredients": ingredientes,
        "number": 2,
        "ranking": 1,
        "ignorePantry": False
    }

    try:
        respuesta = requests.get(
            url,
            params=parametros,
            timeout=10
        )

        respuesta.raise_for_status()
        recetas = respuesta.json()

    except requests.exceptions.RequestException:
        bot.send_message(
            message.chat.id,
            "No pude conectarme con el servicio de recetas, proba mas tarde."
        )
        return

    if not recetas:
        bot.send_message(
            message.chat.id,
            "No encontré recetas con esos ingredientes."
        )
        return

    for receta in recetas:

        receta_id = receta.get("id")

        if not receta_id:
            continue

        url_detalle = (
            f"https://api.spoonacular.com/recipes/"
            f"{receta_id}/information"
        )

        parametros_detalle = {
            "apiKey": SPOONACULAR_API_KEY
        }

        try:
            respuesta_detalle = requests.get(
                url_detalle,
                params=parametros_detalle,
                timeout=10
            )

            respuesta_detalle.raise_for_status()
            detalle = respuesta_detalle.json()

        except requests.exceptions.RequestException:
            continue

        titulo = detalle.get("title", "Sin título")
        imagen = detalle.get("image")
        instrucciones = detalle.get("analyzedInstructions", [])

        texto = f"<b>{titulo}</b>\n\n"

        if instrucciones and instrucciones[0].get("steps"):

            texto += "<b>Preparación:</b>\n"

            for paso in instrucciones[0]["steps"]:
                numero = paso.get("number")
                descripcion = paso.get("step")

                texto += f"{numero}. {descripcion}\n"

        else:
            texto += "No se encontraron instrucciones para esta receta."
            
        if imagen:
            bot.send_photo(
                message.chat.id,
                imagen
            )
            bot.send_message(
                message.chat.id,
                texto,
                parse_mode="HTML"
            )

@bot.message_handler(commands=["dolar"])
def comando_dolar(message):
    registrar(message)

    dolar_actual = pedir_json(URL_DOLAR_ACTUAL)

    if dolar_actual is not None:
        bot.reply_to(message, "Dólar blue: $" + str(dolar_actual["blue"]["value_sell"]))
    else:
        bot.reply_to(message, "No pude consultar la cotización 😕")
        

@bot.message_handler(commands=["historial"])
def comando_historial(message):
    registrar(message)
    chat_id = message.chat.id

    if not os.path.isfile(RUTA_CSV):
        bot.reply_to(message, "Todavía no hay historial 📭")
        return

    with open(RUTA_CSV, "r", encoding="utf-8") as f:
        filas = list(csv.DictReader(f))

    mias = [f for f in filas if f["chat_id"] == str(chat_id)]
    ultimas = mias[-5:]

    if not ultimas:
        bot.reply_to(message, "No encontré consultas tuyas todavía 📭")
        return

    lineas = [f"• {f['timestamp'][11:16]} — {f['mensaje']}" for f in ultimas]
    texto = "🕐 <b>Tus últimas 5 consultas:</b>\n\n" + "\n".join(lineas)

    bot.send_message(chat_id, texto, parse_mode="HTML")


In [ ]:
@bot.message_handler(content_types=["text", "document", "audio"])
def mensaje_de_texto(message):
    registrar(message)

    if message.content_type == "document":
        bot.reply_to(message, "¡Recibí tu documento! Gracias por enviarlo, sin embargo, no puedo leerlo.")
        return
    elif message.content_type == "audio":
        bot.reply_to(message, "¡Qué buen audio! Lo recibí correctamente, sin embargo, no puedo escucharlo.")
        return

    texto = (message.text or "").strip()
    bot.send_chat_action(message.chat.id, "typing")
    time.sleep(2)
    
    if texto.startswith("/"):
        bot.reply_to(message, "No conozco ese comando. Probá con /atajos.")
        return
        
    bot.reply_to(message, "¡Hola! Podes ver los comandos en el menú o usando /atajos")

In [ ]:
from telebot import types

bot.set_my_commands([
    types.BotCommand("start", "Iniciar la conversación"),
    types.BotCommand("atajos", "Ver atajos"),
    types.BotCommand("receta", "Pedir recetas segun ingredientes"),
    types.BotCommand("dolar", "Ver cotización"),
    types.BotCommand("historial", "Ver historial"),
])

print("Menú de comandos configurado.")


In [ ]:
# Ejecutar esta celda al final. Mantener el notebook abierto mientras se prueba el bot.
bot.remove_webhook()
time.sleep(1)
print("Bot escuchando con polling. Para detenerlo: Interrupt Kernel.")
bot.infinity_polling(
    skip_pending=True,
    timeout=20,
    long_polling_timeout=20,
    allowed_updates=["message", "callback_query"],
)